# 작은 AI 루프 데모

이 노트북은 `계획 -> 구현 -> 확정 검증 -> 가벼운 AI 채점 -> 테스트 -> 실패 원인 분석 -> 재시도` 흐름을 직접 보기 위한 학습용 데모입니다.

과제는 일부러 작게 둡니다. 구현 에이전트는 `solution.py`에 `extract_keywords_and_summary(text)` 함수 하나만 만듭니다. 이 함수는 짧은 글을 받아 핵심 키워드 2~4개와 한 줄 요약을 반환해야 합니다.

중요한 원칙:
- 모든 오케스트레이션은 이 노트북 Python 코드 안에서 진행합니다.
- 별도 파이썬 실행 스크립트는 만들지 않습니다.
- hidden 정답은 구현자와 진단자에게 넘기지 않습니다.
- `solution.py`는 노트북에 import하지 않고 subprocess로 격리 실행합니다.
- 마지막 셀에서 `RUN_LOOP = False`로 바꾸기 전에는 실제 루프가 돌지 않습니다.


## 1. 환경 점검

Jupyter에서 실행하면 터미널보다 `PATH`가 좁아질 수 있습니다. 아래 셀은 `codex`, `claude`, Python 실행 경로를 확인하고, 노트북이 항상 homecook 루트에서 동작하도록 맞춥니다.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone, timedelta
from html import escape
import json
import os
import re
import shutil
import statistics
import subprocess
import sys
import time
import textwrap

try:
    from IPython.display import HTML, display
except Exception:
    HTML = None

    def display(value):
        print(value)


def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "package.json").exists() and (candidate / ".git").exists():
            return candidate
    raise RuntimeError("homecook 프로젝트 루트를 찾지 못했습니다.")


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)

KST = timezone(timedelta(hours=9), name="KST")
os.environ["TZ"] = "Asia/Seoul"
if hasattr(time, "tzset"):
    time.tzset()


def kst_now():
    return datetime.now(KST)


def kst_timestamp():
    return kst_now().strftime("%Y-%m-%d %H:%M:%S KST")

path_hints = [
    PROJECT_ROOT / "node_modules" / ".bin",
    Path.home() / ".nvm" / "versions" / "node" / "v22.13.1" / "bin",
    Path.home() / ".local" / "bin",
    Path("/opt/homebrew/bin"),
    Path("/usr/local/bin"),
]
existing_hints = [str(path) for path in path_hints if path.exists()]
os.environ["PATH"] = os.pathsep.join(existing_hints + [os.environ.get("PATH", "")])

print(f"프로젝트 루트: {PROJECT_ROOT}")
print(f"Python: {sys.executable}")
print(f"시간대: {kst_timestamp()}")

required_tools = ["codex", "claude", "git"]
missing = []
for tool in required_tools:
    found = shutil.which(tool)
    print(f"{tool}: {found or 'NOT FOUND'}")
    if not found:
        missing.append(tool)

if missing:
    raise RuntimeError(f"필수 CLI를 찾지 못했습니다: {', '.join(missing)}")


## 2. 데모 과제와 규칙

여기서 과제, 공개 예시, hidden 예시, 점수 기준을 정의합니다. hidden 예시는 노트북 채점 코드만 사용하고, Codex 구현 프롬프트나 Claude 진단 프롬프트에는 넘기지 않습니다.


In [ ]:
MAX_ITER = 3
RUN_TIMEOUT_SECONDS = 5

DEMO_STRICT_FIRST_PASS = True
DEMO_FORCE_FIRST_FAILURE = True

NORMAL_CASE_MIN = 3.5
NORMAL_AVG_MIN = 4.0
STRICT_CASE_MIN = 4.6
STRICT_AVG_MIN = 4.7

RUN_ROOT = PROJECT_ROOT / "notebooks" / "loop_demo_runs"
RUN_ROOT.mkdir(parents=True, exist_ok=True)

TASK_NAME = "짧은 글에서 핵심 키워드와 한 줄 요약 뽑기"
SOLUTION_FILENAME = "solution.py"
FUNCTION_NAME = "extract_keywords_and_summary"

PUBLIC_EXAMPLES = [
    {
        "id": "public-01",
        "text": "비가 많이 와서 오후 축제가 실내 체육관으로 옮겨졌다.",
        "good_output": {
            "keywords": ["비", "오후 축제", "실내 체육관"],
            "summary": "비 때문에 오후 축제가 실내 체육관으로 변경됐다.",
        },
    },
    {
        "id": "public-02",
        "text": "동네 카페가 일회용 컵을 줄이려고 개인 컵 손님에게 할인을 제공한다.",
        "good_output": {
            "keywords": ["동네 카페", "일회용 컵", "개인 컵 할인"],
            "summary": "동네 카페가 개인 컵 할인을 통해 일회용 컵 사용을 줄이려 한다.",
        },
    },
]

HIDDEN_CASES = [
    {
        "id": "hidden-01",
        "text": "폭우로 아침 지하철 운행이 늦어지자 시는 배수 시설을 긴급 점검했다.",
        "expected_keywords": ["폭우", "지하철 지연", "배수 시설 점검"],
        "expected_summary": "폭우로 지하철이 지연되어 시가 배수 시설을 긴급 점검했다.",
    },
    {
        "id": "hidden-02",
        "text": "동네 도서관은 직장인 이용자를 위해 다음 달부터 평일 야간 운영을 두 시간 늘린다.",
        "expected_keywords": ["동네 도서관", "직장인 이용자", "야간 운영 확대"],
        "expected_summary": "동네 도서관이 직장인을 위해 평일 야간 운영 시간을 늘린다.",
    },
    {
        "id": "hidden-03",
        "text": "학교 급식실은 지역 농산물 사용을 늘려 식재료 운송 거리를 줄이기로 했다.",
        "expected_keywords": ["학교 급식", "지역 농산물", "운송 거리 감소"],
        "expected_summary": "학교 급식실이 지역 농산물을 더 써서 식재료 운송 거리를 줄이려 한다.",
    },
]

FIXED_SAFE_HINTS = [
    "키워드는 구체 명사를 우선하라.",
    "요약은 누가/무엇/왜/결과 중 핵심을 한 문장으로 담아라.",
    "너무 일반적인 단어만 뽑지 말라.",
    "입력 문장을 그대로 복붙하지 말고 중요한 정보만 압축하라.",
]

print(f"반복 상한 MAX_ITER = {MAX_ITER}")
print(f"정상 기준: 모든 case_score >= {NORMAL_CASE_MIN}, average_score >= {NORMAL_AVG_MIN}")
print(f"1회차 strict 기준: 모든 case_score >= {STRICT_CASE_MIN}, average_score >= {STRICT_AVG_MIN}")


## 3. 공통 도구 함수

아래 함수들은 로그 저장, JSON 추출, 에이전트 실행, 목차 생성을 담당합니다.


In [ ]:
ANSI_RE = re.compile(r"\x1b\[[0-9;]*m")
UTC_TIMESTAMP_RE = re.compile(r"\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}(?:\.\d+)?Z")


def strip_ansi(text):
    return ANSI_RE.sub("", text or "")


def utc_timestamp_to_kst(match):
    raw = match.group(0)
    dt = datetime.fromisoformat(raw.replace("Z", "+00:00"))
    return dt.astimezone(KST).strftime("%Y-%m-%d %H:%M:%S KST")


def convert_utc_timestamps_to_kst(text):
    return UTC_TIMESTAMP_RE.sub(utc_timestamp_to_kst, text)


def print_stage(message):
    print("\n" + "=" * 80)
    print(f"[{kst_timestamp()}] {message}")
    print("=" * 80)


def write_text(path, text):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def write_json(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


def read_text(path):
    return path.read_text(encoding="utf-8") if path.exists() else ""


def read_json(path, default=None):
    if default is None:
        default = {}
    try:
        return json.loads(Path(path).read_text(encoding="utf-8"))
    except Exception:
        return default


def run_agent_command(cmd, prompt, log_path, cwd=None):
    """AI CLI를 실행하고 stdout/stderr를 로그로 남긴다."""
    chunks = []
    full_cmd = list(cmd) + [prompt]
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("w", encoding="utf-8") as log:
        proc = subprocess.Popen(
            full_cmd,
            cwd=str(cwd or PROJECT_ROOT),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=os.environ.copy(),
        )
        for line in proc.stdout:
            display_line = convert_utc_timestamps_to_kst(line)
            print(display_line, end="")
            log.write(display_line)
            chunks.append(display_line)
        proc.wait()
    return {"returncode": proc.returncode, "output": "".join(chunks)}


def extract_json_object(text):
    """모델 응답에서 채점 JSON object를 꺼낸다.

    Codex CLI 출력에는 경고, 토큰 사용량, 중복 JSON이 섞일 수 있다.
    그래서 '처음 {부터 마지막 }까지'가 아니라, 실제로 파싱 가능한 JSON object들을
    모두 찾아서 cases/average_score가 있는 마지막 object를 우선 사용한다.
    """
    cleaned = strip_ansi(text)
    decoder = json.JSONDecoder()
    candidates = []

    for fenced in re.finditer(r"```(?:json)?\s*(\{.*?\})\s*```", cleaned, re.DOTALL):
        try:
            obj = json.loads(fenced.group(1))
            if isinstance(obj, dict):
                candidates.append(obj)
        except json.JSONDecodeError:
            pass

    for match in re.finditer(r"\{", cleaned):
        fragment = cleaned[match.start():]
        try:
            obj, _ = decoder.raw_decode(fragment)
        except json.JSONDecodeError:
            continue
        if isinstance(obj, dict):
            candidates.append(obj)

    for obj in reversed(candidates):
        if "cases" in obj and "average_score" in obj:
            return obj
    if candidates:
        return candidates[-1]
    raise ValueError("JSON object를 찾지 못했습니다.")


def case_thresholds(iteration):
    if iteration == 1 and DEMO_STRICT_FIRST_PASS:
        return {
            "mode": "strict-first-pass",
            "case_min": STRICT_CASE_MIN,
            "average_min": STRICT_AVG_MIN,
        }
    return {
        "mode": "normal",
        "case_min": NORMAL_CASE_MIN,
        "average_min": NORMAL_AVG_MIN,
    }


def create_run_dir():
    timestamp = kst_now().strftime("%Y%m%d-%H%M%S")
    run_dir = RUN_ROOT / timestamp
    run_dir.mkdir(parents=True, exist_ok=False)
    return run_dir


def update_readme(run_dir, rows, final_status="running"):
    lines = [
        "# AI 루프 데모 실행 목차",
        "",
        f"- 과제: {TASK_NAME}",
        f"- 반복 상한: {MAX_ITER}",
        f"- 상태: {final_status}",
        "- HTML 대시보드: [dashboard.html](dashboard.html)",
        "",
        "## 반복별 목차",
        "",
        "| 반복 | 결과 | 기준 | 상세 로그 |",
        "|---:|---|---|---|",
    ]
    for row in rows:
        links = [
            f"[계획]({row['dir']}/01_plan.md)",
            f"[구현]({row['dir']}/02_implementation.log)",
            f"[확정검증]({row['dir']}/03_deterministic_verify.json)",
            f"[AI채점]({row['dir']}/04_ai_grade.private.json)",
            f"[판정]({row['dir']}/05_decision.json)",
        ]
        if row.get("diagnosis"):
            links.append(f"[진단]({row['dir']}/06_failure_diagnosis.md)")
        lines.append(
            f"| {row['iteration']} | {row['result']} | {row['threshold_mode']} | "
            + ", ".join(links)
            + " |"
        )
    lines.extend([
        "",
        "## 읽는 법",
        "",
        "1. `dashboard.html`에서 전체 흐름과 반복별 결과를 먼저 봅니다.",
        "2. `01_plan.md`에서 그 반복이 무엇을 하려 했는지 봅니다.",
        "3. `02_implementation.log`에서 Codex가 무엇을 했는지 봅니다.",
        "4. `03_deterministic_verify.json`에서 확정 검증 결과를 봅니다.",
        "5. `04_ai_grade.private.json`은 hidden 기준을 포함할 수 있는 private 채점 로그입니다.",
        "6. `05_decision.json`에서 최종 합격/불합격 기준 적용 결과를 봅니다.",
        "7. 실패한 반복은 `06_failure_diagnosis.md`와 `feedback_for_next_iter.md`를 이어서 봅니다.",
    ])
    write_text(run_dir / "README.md", "\n".join(lines) + "\n")


STYLE_BLOCK = """
<style>
.ai-loop-ui {
  --paper: #fbfaf7;
  --ink: #151515;
  --muted: #62615d;
  --line: #d9d3c7;
  --soft: #f1ede4;
  --green: #0f7a55;
  --red: #b33b2e;
  --amber: #9a6a00;
  --blue: #275d8c;
  --violet: #6d5278;
  font-family: ui-serif, Georgia, "Times New Roman", serif;
  color: var(--ink);
  background: var(--paper);
  border: 1px solid var(--line);
  border-radius: 8px;
  padding: 22px;
  margin: 14px 0;
  box-shadow: 0 10px 30px rgba(0, 0, 0, 0.07);
}
.ai-loop-ui * { box-sizing: border-box; }
.ai-loop-ui h2, .ai-loop-ui h3, .ai-loop-ui h4 { margin: 0; letter-spacing: 0; }
.ai-loop-ui p { margin: 0; }
.ai-loop-top { display: flex; justify-content: space-between; gap: 18px; align-items: flex-start; border-bottom: 1px solid var(--line); padding-bottom: 16px; }
.ai-loop-title { font-size: 28px; line-height: 1.15; }
.ai-loop-subtitle { color: var(--muted); margin-top: 8px; line-height: 1.5; max-width: 760px; }
.ai-loop-eyebrow { font-family: ui-monospace, SFMono-Regular, Menlo, monospace; font-size: 11px; letter-spacing: 0; color: var(--muted); text-transform: uppercase; margin-bottom: 8px; }
.ai-loop-grid { display: grid; grid-template-columns: repeat(4, minmax(0, 1fr)); gap: 10px; margin-top: 16px; }
.ai-loop-flow { display: grid; grid-template-columns: repeat(7, minmax(132px, 1fr)); gap: 8px; margin-top: 18px; overflow-x: auto; padding-bottom: 4px; }
.ai-loop-card, .ai-loop-stage, .ai-loop-panel { background: #fffdf8; border: 1px solid var(--line); border-radius: 8px; padding: 13px; }
.ai-loop-stage { min-height: 155px; position: relative; }
.ai-loop-stage::after { content: ""; position: absolute; right: -8px; top: 50%; width: 8px; border-top: 1px solid var(--line); }
.ai-loop-stage:last-child::after { display: none; }
.ai-loop-stage-num { font-family: ui-monospace, SFMono-Regular, Menlo, monospace; font-size: 11px; color: var(--muted); }
.ai-loop-stage h4 { margin-top: 6px; font-size: 16px; }
.ai-loop-role { margin-top: 8px; font-family: ui-monospace, SFMono-Regular, Menlo, monospace; font-size: 12px; color: var(--blue); }
.ai-loop-desc { margin-top: 8px; color: var(--muted); line-height: 1.45; font-size: 13px; }
.ai-loop-badge-row { display: flex; flex-wrap: wrap; gap: 6px; margin-top: 10px; }
.ai-loop-badge { display: inline-flex; align-items: center; border: 1px solid var(--line); border-radius: 999px; padding: 3px 8px; font-family: ui-monospace, SFMono-Regular, Menlo, monospace; font-size: 11px; background: var(--soft); color: var(--ink); white-space: nowrap; }
.ai-loop-badge.ok { color: var(--green); border-color: rgba(15, 122, 85, .35); background: rgba(15, 122, 85, .08); }
.ai-loop-badge.fail { color: var(--red); border-color: rgba(179, 59, 46, .35); background: rgba(179, 59, 46, .08); }
.ai-loop-badge.warn { color: var(--amber); border-color: rgba(154, 106, 0, .35); background: rgba(154, 106, 0, .08); }
.ai-loop-badge.info { color: var(--blue); border-color: rgba(39, 93, 140, .35); background: rgba(39, 93, 140, .08); }
.ai-loop-section { margin-top: 20px; }
.ai-loop-section h3 { font-size: 19px; margin-bottom: 10px; }
.ai-loop-test-list { display: grid; grid-template-columns: repeat(2, minmax(0, 1fr)); gap: 8px; margin-top: 10px; }
.ai-loop-test { border-left: 3px solid var(--line); padding: 9px 10px; background: #fffdf8; line-height: 1.45; font-size: 13px; }
.ai-loop-test strong { display: block; margin-bottom: 3px; }
.ai-loop-run-list { display: grid; gap: 12px; margin-top: 12px; }
.ai-loop-iter { border: 1px solid var(--line); border-radius: 8px; background: #fffdf8; overflow: hidden; }
.ai-loop-iter-head { display: flex; justify-content: space-between; gap: 12px; align-items: center; padding: 14px; border-bottom: 1px solid var(--line); background: #f5f1e8; }
.ai-loop-iter-title { font-size: 18px; font-weight: 700; }
.ai-loop-iter-body { padding: 14px; display: grid; grid-template-columns: 1.2fr .8fr; gap: 14px; }
.ai-loop-step-table { width: 100%; border-collapse: collapse; font-size: 13px; }
.ai-loop-step-table th, .ai-loop-step-table td { text-align: left; border-bottom: 1px solid var(--line); padding: 8px 6px; vertical-align: top; }
.ai-loop-step-table th { color: var(--muted); font-weight: 600; }
.ai-loop-step-table a { color: var(--blue); text-decoration: none; border-bottom: 1px solid rgba(39, 93, 140, .35); }
.ai-loop-metrics { display: grid; grid-template-columns: repeat(2, minmax(0, 1fr)); gap: 8px; }
.ai-loop-metric { background: var(--soft); border-radius: 8px; padding: 10px; border: 1px solid var(--line); }
.ai-loop-metric-label { font-size: 11px; color: var(--muted); font-family: ui-monospace, SFMono-Regular, Menlo, monospace; }
.ai-loop-metric-value { margin-top: 4px; font-size: 18px; font-weight: 700; }
.ai-loop-note { color: var(--muted); line-height: 1.5; font-size: 13px; }
.ai-loop-small { font-size: 12px; color: var(--muted); line-height: 1.45; }
@media (max-width: 900px) {
  .ai-loop-top, .ai-loop-iter-head { flex-direction: column; align-items: flex-start; }
  .ai-loop-grid, .ai-loop-test-list, .ai-loop-iter-body, .ai-loop-metrics { grid-template-columns: 1fr; }
  .ai-loop-flow { grid-template-columns: repeat(7, minmax(150px, 1fr)); }
}
</style>
"""


def html_text(value):
    return escape("" if value is None else str(value))


def badge(label, tone=""):
    tone_class = f" {tone}" if tone else ""
    return f'<span class="ai-loop-badge{tone_class}">{html_text(label)}</span>'


def file_link(path, label):
    path = Path(path)
    if not path.exists():
        return f'<span class="ai-loop-small">{html_text(label)} 없음</span>'
    return f'<a href="{path.resolve().as_uri()}">{html_text(label)}</a>'


def stage_cards_html():
    stages = [
        ("01", "계획", "Claude", "첫 회차는 전체 계획, 다음 회차부터는 직전 피드백 반영 계획만 작성합니다.", "01_plan.md"),
        ("02", "구현", "Codex", "공개 예시와 안전 피드백만 보고 solution.py 함수 하나를 고칩니다.", "02_implementation.log"),
        ("03", "확정 검증", "Notebook Python", "파일 존재, hidden 문구 하드코딩, subprocess 실행, 출력 구조를 코드로 확인합니다.", "03_deterministic_verify.json"),
        ("04", "의미 채점", "Codex Spark", "키워드와 요약이 기준 답안과 의미상 가까운지 0~5점으로 채점합니다.", "04_ai_grade.private.json"),
        ("05", "테스트 판정", "Notebook Python", "case_score와 average_score를 명확한 기준에 대입해 통과 여부를 결정합니다.", "05_decision.json"),
        ("06", "실패 진단", "Claude", "hidden 정답 없이 코드와 집계 점수만 보고 다음 수정 방향을 씁니다.", "06_failure_diagnosis.md"),
        ("07", "재시도", "Notebook Python", "안전 피드백만 다음 ITER에 넘기고 최대 MAX_ITER까지만 반복합니다.", "feedback_for_next_iter.md"),
    ]
    return "\n".join(
        f"""
        <article class="ai-loop-stage">
          <div class="ai-loop-stage-num">ITER STEP {num}</div>
          <h4>{html_text(name)}</h4>
          <div class="ai-loop-role">{html_text(role)}</div>
          <p class="ai-loop-desc">{html_text(desc)}</p>
          <div class="ai-loop-badge-row">{badge(output, 'info')}</div>
        </article>
        """
        for num, name, role, desc, output in stages
    )


def display_loop_overview():
    html = f"""
    {STYLE_BLOCK}
    <section class="ai-loop-ui">
      <div class="ai-loop-top">
        <div>
          <div class="ai-loop-eyebrow">Jupyter AI Loop Demo</div>
          <h2 class="ai-loop-title">ITER 하나는 이렇게 돕니다</h2>
          <p class="ai-loop-subtitle">
            이 화면은 모델별 역할과 테스트 위치를 먼저 보여줍니다. 아래 루프를 읽은 뒤 실행 로그를 보면
            Claude, Codex, Codex Spark, Notebook Python이 각각 어디에 쓰이는지 따라가기 쉽습니다.
          </p>
        </div>
        <div class="ai-loop-badge-row">
          {badge(f'MAX_ITER = {MAX_ITER}', 'info')}
          {badge('hidden 정답 누수 차단', 'warn')}
          {badge('subprocess 격리 실행', 'ok')}
        </div>
      </div>
      <div class="ai-loop-flow">{stage_cards_html()}</div>
      <div class="ai-loop-section">
        <h3>테스트는 두 종류입니다</h3>
        <div class="ai-loop-test-list">
          <div class="ai-loop-test"><strong>확정 검증</strong>AI 판단 없이 Python 코드가 확인합니다. 함수가 실행되는지, 출력이 비지 않는지, hidden 문장을 코드에 박아넣지 않았는지 봅니다.</div>
          <div class="ai-loop-test"><strong>의미 채점</strong>정답과 글자가 완전히 같지 않아도 의미가 맞는지 Codex Spark가 0~5점으로 평가합니다.</div>
          <div class="ai-loop-test"><strong>최종 판정</strong>AI가 합격을 선언하지 않습니다. Notebook Python이 점수 기준을 계산해서 PASS/FAIL을 정합니다.</div>
          <div class="ai-loop-test"><strong>실패 피드백</strong>Claude는 hidden 정답을 보지 않고 코드와 집계 점수만 보고 다음 구현 방향을 적습니다.</div>
        </div>
      </div>
    </section>
    """
    display(HTML(html) if HTML else html)


def latest_run_dir():
    if not RUN_ROOT.exists():
        return None
    candidates = [path for path in RUN_ROOT.iterdir() if path.is_dir()]
    if not candidates:
        return None
    return max(candidates, key=lambda path: path.stat().st_mtime)


def run_status_from_readme(run_dir):
    readme = read_text(run_dir / "README.md")
    match = re.search(r"^- 상태:\s*(.+)$", readme, re.MULTILINE)
    return match.group(1).strip() if match else "unknown"


def grade_summary_for_dashboard(iter_dir):
    grade_path = iter_dir / "04_ai_grade.private.json"
    grade = read_json(grade_path, default={})
    source = "04_ai_grade.private.json"
    parse_error = grade.get("parse_error")
    recovered = None

    if grade.get("cases"):
        recovered = grade
    elif parse_error:
        raw_path = iter_dir / "04_ai_grade.raw.private.log"
        try:
            recovered = normalize_grade(extract_json_object(read_text(raw_path)))
            source = "raw private log에서 집계 복구"
        except Exception:
            recovered = grade
    else:
        recovered = grade

    cases = recovered.get("cases", []) if isinstance(recovered, dict) else []
    case_scores = [float(case.get("case_score", 0.0)) for case in cases]
    return {
        "source": source,
        "parse_error": parse_error,
        "case_count": len(cases),
        "min_case_score": min(case_scores) if case_scores else 0.0,
        "average_score": float(recovered.get("average_score", 0.0)) if isinstance(recovered, dict) else 0.0,
    }


def deterministic_checks_html(deterministic):
    checks = [
        ("solution.py 존재", deterministic.get("solution_exists") is True),
        ("hidden 문구 하드코딩 없음", deterministic.get("forbidden_hidden_text_found") is False),
        (f"{RUN_TIMEOUT_SECONDS}초 안에 subprocess 종료", deterministic.get("runner_timeout") is False),
        ("runner 종료코드 0", deterministic.get("runner_returncode") == 0),
        ("반환 구조 정상", deterministic.get("structure_ok") is True),
        ("hidden case 출력 개수 일치", deterministic.get("output_count") == deterministic.get("hidden_case_count") and deterministic.get("output_count") is not None),
    ]
    return "".join(
        f"<div>{badge('PASS' if ok else 'FAIL', 'ok' if ok else 'fail')} <span>{html_text(label)}</span></div>"
        for label, ok in checks
    )


def iteration_steps_table(iter_dir, decision):
    failed = not decision.get("passed", False)
    rows = [
        ("Claude", "계획", iter_dir / "01_plan.md", "전체 계획 또는 피드백 반영 계획"),
        ("Codex", "구현", iter_dir / "02_implementation.log", "solution.py 함수 구현"),
        ("Notebook Python", "확정 검증", iter_dir / "03_deterministic_verify.json", "실행 가능성과 꼼수 방지 확인"),
        ("Codex Spark", "의미 채점", iter_dir / "04_ai_grade.private.json", "키워드/요약 의미 점수"),
        ("Notebook Python", "최종 판정", iter_dir / "05_decision.json", "명시 기준으로 PASS/FAIL 결정"),
    ]
    if failed:
        rows.append(("Claude", "실패 진단", iter_dir / "06_failure_diagnosis.md", "hidden 정답 없이 원인과 방향 정리"))
    body = "".join(
        f"""
        <tr>
          <td>{html_text(model)}</td>
          <td>{html_text(step)}</td>
          <td>{file_link(path, path.name)}</td>
          <td>{html_text(desc)}</td>
        </tr>
        """
        for model, step, path, desc in rows
    )
    return f"""
    <table class="ai-loop-step-table">
      <thead><tr><th>담당</th><th>단계</th><th>로그</th><th>무슨 일</th></tr></thead>
      <tbody>{body}</tbody>
    </table>
    """


def iteration_card_html(iter_dir):
    decision = read_json(iter_dir / "05_decision.json", default={})
    deterministic = read_json(iter_dir / "03_deterministic_verify.json", default={})
    score = grade_summary_for_dashboard(iter_dir)
    iteration = decision.get("iteration") or iter_dir.name.replace("iteration-", "")
    passed = bool(decision.get("passed"))
    tone = "ok" if passed else "fail"
    result_label = "PASS" if passed else "FAIL"
    threshold_mode = decision.get("threshold_mode", "unknown")
    parse_note = ""
    if score.get("parse_error"):
        parse_note = (
            f"<p class='ai-loop-small'>주의: 저장된 채점 JSON 파싱에 실패했습니다. "
            f"이 카드는 hidden 내용을 보여주지 않고 raw private log에서 집계 점수만 복구해 표시합니다.</p>"
        )
    return f"""
    <article class="ai-loop-iter">
      <div class="ai-loop-iter-head">
        <div>
          <div class="ai-loop-iter-title">ITER {html_text(iteration)}</div>
          <div class="ai-loop-badge-row">
            {badge(result_label, tone)}
            {badge(threshold_mode, 'info')}
            {badge('확정 검증 통과' if deterministic.get('passed') else '확정 검증 실패', 'ok' if deterministic.get('passed') else 'fail')}
          </div>
        </div>
        <div class="ai-loop-small">{html_text(iter_dir.name)} · {html_text(score['source'])}</div>
      </div>
      <div class="ai-loop-iter-body">
        <div>
          {iteration_steps_table(iter_dir, decision)}
        </div>
        <div>
          <div class="ai-loop-metrics">
            <div class="ai-loop-metric"><div class="ai-loop-metric-label">판정 평균</div><div class="ai-loop-metric-value">{decision.get('average_score', 0.0):.2f}</div></div>
            <div class="ai-loop-metric"><div class="ai-loop-metric-label">채점 원문 평균</div><div class="ai-loop-metric-value">{score['average_score']:.2f}</div></div>
            <div class="ai-loop-metric"><div class="ai-loop-metric-label">최저 case</div><div class="ai-loop-metric-value">{score['min_case_score']:.2f}</div></div>
            <div class="ai-loop-metric"><div class="ai-loop-metric-label">필요 기준</div><div class="ai-loop-metric-value">{decision.get('case_min_required', 0.0):.1f} / {decision.get('average_min_required', 0.0):.1f}</div></div>
          </div>
          <div class="ai-loop-section">
            <h3>확정 검증 체크</h3>
            <div class="ai-loop-small">{deterministic_checks_html(deterministic)}</div>
          </div>
          {parse_note}
        </div>
      </div>
    </article>
    """


def build_run_dashboard_html(run_dir):
    run_dir = Path(run_dir)
    status = run_status_from_readme(run_dir)
    iter_dirs = sorted(path for path in run_dir.glob("iteration-*") if path.is_dir())
    pass_count = sum(1 for path in iter_dirs if read_json(path / "05_decision.json", default={}).get("passed"))
    fail_count = len(iter_dirs) - pass_count
    cards = "\n".join(iteration_card_html(path) for path in iter_dirs)
    if not cards:
        cards = "<p class='ai-loop-note'>아직 실행된 ITER가 없습니다.</p>"
    html = f"""
    {STYLE_BLOCK}
    <section class="ai-loop-ui">
      <div class="ai-loop-top">
        <div>
          <div class="ai-loop-eyebrow">Run Dashboard</div>
          <h2 class="ai-loop-title">모델별 루프 실행 흐름</h2>
          <p class="ai-loop-subtitle">
            이 HTML은 실행 폴더의 로그를 읽어서 ITER별 담당 모델, 테스트 종류, 점수 기준, 결과 파일을 한 화면에 묶어 보여줍니다.
            private 채점 로그의 hidden 문장과 기준 답안은 화면에 표시하지 않습니다.
          </p>
        </div>
        <div class="ai-loop-badge-row">
          {badge(f'상태: {status}', 'ok' if status == 'passed' else 'warn' if status == 'running' else 'fail')}
          {badge(f'실행 ITER: {len(iter_dirs)} / {MAX_ITER}', 'info')}
          {badge(f'PASS {pass_count}', 'ok')}
          {badge(f'FAIL {fail_count}', 'fail' if fail_count else 'ok')}
        </div>
      </div>
      <div class="ai-loop-grid">
        <div class="ai-loop-card"><div class="ai-loop-metric-label">과제</div><div class="ai-loop-note">{html_text(TASK_NAME)}</div></div>
        <div class="ai-loop-card"><div class="ai-loop-metric-label">정상 기준</div><div class="ai-loop-note">모든 case ≥ {NORMAL_CASE_MIN}, 평균 ≥ {NORMAL_AVG_MIN}</div></div>
        <div class="ai-loop-card"><div class="ai-loop-metric-label">1회차 strict 기준</div><div class="ai-loop-note">모든 case ≥ {STRICT_CASE_MIN}, 평균 ≥ {STRICT_AVG_MIN}</div></div>
        <div class="ai-loop-card"><div class="ai-loop-metric-label">실행 폴더</div><div class="ai-loop-note">{file_link(run_dir / 'README.md', run_dir.name)}</div></div>
      </div>
      <div class="ai-loop-section">
        <h3>한 번의 ITER 흐름</h3>
        <div class="ai-loop-flow">{stage_cards_html()}</div>
      </div>
      <div class="ai-loop-section">
        <h3>반복별 결과</h3>
        <div class="ai-loop-run-list">{cards}</div>
      </div>
      <div class="ai-loop-section">
        <h3>테스트 기준 해석</h3>
        <div class="ai-loop-test-list">
          <div class="ai-loop-test"><strong>확정 검증 통과</strong>함수가 subprocess에서 끝나고, 반환 구조가 맞고, hidden 문장을 코드에 그대로 넣지 않았다는 뜻입니다.</div>
          <div class="ai-loop-test"><strong>case_score</strong>키워드 점수와 요약 점수 중 낮은 값을 씁니다. 둘 중 하나만 좋아도 통과하지 못하게 하려는 장치입니다.</div>
          <div class="ai-loop-test"><strong>average_score</strong>모든 hidden case의 case_score 평균입니다. 최종 합격은 모든 case 기준과 평균 기준을 둘 다 넘겨야 합니다.</div>
          <div class="ai-loop-test"><strong>진단 피드백</strong>다음 ITER에는 hidden 정답이 아니라 집계 점수, 고정 힌트, 코드 기반 원인만 전달합니다.</div>
        </div>
      </div>
    </section>
    """
    return html


def ensure_dashboard_link_in_readme(run_dir):
    readme_path = Path(run_dir) / "README.md"
    if not readme_path.exists():
        return
    readme = read_text(readme_path)
    if "dashboard.html" in readme:
        return
    readme = re.sub(
        r"(- 상태: .+\n)",
        r"\1- HTML 대시보드: [dashboard.html](dashboard.html)\n",
        readme,
        count=1,
    )
    write_text(readme_path, readme)


def write_run_dashboard(run_dir):
    run_dir = Path(run_dir)
    ensure_dashboard_link_in_readme(run_dir)
    html = "<!doctype html>\n<meta charset='utf-8'>\n<title>AI Loop Dashboard</title>\n" + build_run_dashboard_html(run_dir)
    write_text(run_dir / "dashboard.html", html)
    return run_dir / "dashboard.html"


def display_run_dashboard(run_dir=None):
    run_dir = Path(run_dir) if run_dir else latest_run_dir()
    if not run_dir:
        print("아직 loop_demo_runs 실행 폴더가 없습니다.")
        return None
    dashboard_path = write_run_dashboard(run_dir)
    display(HTML(build_run_dashboard_html(run_dir)) if HTML else dashboard_path)
    print(f"HTML 대시보드 파일: {dashboard_path}")
    return dashboard_path


def display_iteration_banner(iteration):
    thresholds = case_thresholds(iteration)
    html = f"""
    {STYLE_BLOCK}
    <section class="ai-loop-ui">
      <div class="ai-loop-top">
        <div>
          <div class="ai-loop-eyebrow">ITER {iteration} / {MAX_ITER}</div>
          <h2 class="ai-loop-title">이번 반복에서 볼 흐름</h2>
          <p class="ai-loop-subtitle">
            기준은 {html_text(thresholds['mode'])}입니다. 이 반복은 계획, 구현, 확정 검증, 의미 채점, 판정, 실패 진단 순서로 진행됩니다.
          </p>
        </div>
        <div class="ai-loop-badge-row">
          {badge(f"case ≥ {thresholds['case_min']}", 'info')}
          {badge(f"average ≥ {thresholds['average_min']}", 'info')}
        </div>
      </div>
      <div class="ai-loop-flow">{stage_cards_html()}</div>
    </section>
    """
    display(HTML(html) if HTML else html)


def display_iteration_result(iter_dir):
    html = f"{STYLE_BLOCK}<section class='ai-loop-ui'><div class='ai-loop-run-list'>{iteration_card_html(Path(iter_dir))}</div></section>"
    display(HTML(html) if HTML else html)


## 4. 루프 지도

아래 HTML은 한 번의 `ITER`가 어떤 모델과 테스트를 거치는지 먼저 보여줍니다. 실행 로그를 읽기 전에 이 그림을 보면 전체 흐름을 잡기 쉽습니다.


In [ ]:
display_loop_overview()

## 5. 에이전트 프롬프트

역할별 프롬프트를 분리합니다. 구현자와 진단자에게는 hidden 입력/정답/실제 출력이 가지 않도록 합니다.


In [ ]:
CLAUDE_CMD = ["claude", "-p", "--permission-mode", "plan"]
CODEX_IMPLEMENT_CMD_BASE = ["codex", "exec", "--sandbox", "workspace-write"]
CODEX_GRADER_CMD = [
    "codex",
    "exec",
    "--model",
    "gpt-5.3-codex-spark",
    "--sandbox",
    "read-only",
    "--ephemeral",
]


def public_examples_text():
    return json.dumps(PUBLIC_EXAMPLES, ensure_ascii=False, indent=2)


def build_initial_plan_prompt():
    return f"""
너는 이 노트북 데모의 계획 담당자다.
아주 짧고 읽기 쉬운 구현 계획만 작성하라.

과제: {TASK_NAME}
구현 파일: solution.py
필수 함수: {FUNCTION_NAME}(text)
반환 형식: {{"keywords": [문자열 2~4개], "summary": "한 줄 요약"}}

공개 예시:
{public_examples_text()}

hidden 테스트나 정답은 알 수 없다고 가정하라.
출력은 마크다운으로, 다음 세 문단만 써라:
1. 접근
2. 구현 체크리스트
3. 조심할 점
""".strip()


def build_replan_prompt(previous_feedback):
    return f"""
너는 이 노트북 데모의 계획 담당자다.
이번은 전체 재계획이 아니라 직전 실패 피드백을 반영한 짧은 수정 계획만 작성하라.

과제: {TASK_NAME}
반환 형식: {{"keywords": [문자열 2~4개], "summary": "한 줄 요약"}}

직전 피드백:
{previous_feedback}

출력은 마크다운으로, 다음 세 문단만 써라:
1. 이번 반복에서 바꿀 점
2. 유지할 점
3. 구현자에게 줄 주의사항
""".strip()


def build_implementation_prompt(plan_text, feedback_text):
    return f"""
너는 구현 담당 Codex다.
현재 작업 디렉터리의 solution.py 하나만 수정하라. 다른 파일은 만들거나 수정하지 마라.

과제: {TASK_NAME}
필수 함수 이름: {FUNCTION_NAME}
입력: 짧은 한국어 문자열 text
반환: dict, 정확히 다음 구조를 지켜라.
{{"keywords": [문자열 2~4개], "summary": "한 줄 요약"}}

공개 예시:
{public_examples_text()}

계획:
{plan_text}

직전 피드백:
{feedback_text or "없음"}

구현 지침:
- hidden 테스트 문장을 알 수 없다고 가정하라.
- 공개 예시만 외워서 처리하지 마라.
- 외부 패키지를 추가하지 마라.
- 간단한 규칙 기반 구현으로 충분하다.
- 키워드는 너무 일반적인 단어보다 구체적인 명사/명사구를 우선하라.
- 요약은 한 문장으로 작성하라.
""".strip()


def build_grading_prompt(grading_payload):
    return f"""
너는 가벼운 의미 채점자다. 반드시 JSON object 하나만 출력하라.

점수 척도는 모든 항목이 0.0~5.0 실수다.
- keyword_score: 핵심 키워드가 기준 답안과 의미상 맞는 정도
- summary_score: 한 줄 요약이 기준 답안과 의미상 맞는 정도
- case_score: 반드시 min(keyword_score, summary_score)로 계산

합산 규칙:
- case_score = min(keyword_score, summary_score)
- average_score = 모든 case_score의 산술 평균

채점할 데이터:
{json.dumps(grading_payload, ensure_ascii=False, indent=2)}

출력 JSON 형식:
{{
  "cases": [
    {{
      "id": "case id",
      "keyword_score": 0.0,
      "summary_score": 0.0,
      "case_score": 0.0,
      "reason": "짧은 이유"
    }}
  ],
  "average_score": 0.0
}}
""".strip()


def build_diagnosis_prompt(solution_code, deterministic_safe, decision_safe):
    return f"""
너는 실패 원인 분석 담당 Claude다.
아래 정보만 보고 코드 기반 원인과 다음 수정 방향을 설명하라.

절대 금지:
- hidden 입력 문장 추측 또는 언급
- hidden 기준 정답 추측 또는 언급
- 실제 hidden 출력 추측 또는 언급
- 케이스별 누락 항목 추측 또는 언급

제공 정보:
1) 현재 solution.py 코드
```python
{solution_code}
```

2) 확정 검증 결과(안전 요약)
```json
{json.dumps(deterministic_safe, ensure_ascii=False, indent=2)}
```

3) 점수 집계와 판정(안전 요약)
```json
{json.dumps(decision_safe, ensure_ascii=False, indent=2)}
```

4) 사전 정의된 고정 힌트. 이 힌트는 hidden 결과에서 만든 것이 아니다.
{chr(10).join(f"- {hint}" for hint in FIXED_SAFE_HINTS)}

출력은 마크다운으로 작성하라:
- 실패 원인 가설
- 다음 반복 수정 방향
- 구현자에게 넘길 짧은 피드백
""".strip()


## 6. 확정 검증과 점수 판정

이 단계는 AI가 아니라 Python 코드가 확정적으로 확인합니다. `solution.py`는 노트북에 import하지 않고 별도 프로세스에서 실행합니다.


In [ ]:
STARTER_SOLUTION = (
    "def extract_keywords_and_summary(text):\n"
    "    \"\"\"TODO: Codex가 이 함수를 구현합니다.\"\"\"\n"
    "    return {\"keywords\": [], \"summary\": \"\"}\n"
)


def copy_previous_solution(previous_solution, next_solution):
    if previous_solution and previous_solution.exists():
        next_solution.write_text(previous_solution.read_text(encoding="utf-8"), encoding="utf-8")
    else:
        next_solution.write_text(STARTER_SOLUTION, encoding="utf-8")


def forbidden_hidden_strings():
    values = []
    for case in HIDDEN_CASES:
        values.append(case["text"])
        values.append(case["expected_summary"])
    return values


def build_runner_code(solution_path):
    cases_for_runner = [{"id": case["id"], "text": case["text"]} for case in HIDDEN_CASES]
    return f"""
import importlib.util
import json
import sys
from pathlib import Path

solution_path = Path({str(solution_path)!r})
spec = importlib.util.spec_from_file_location("student_solution", solution_path)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

fn = getattr(module, {FUNCTION_NAME!r}, None)
if not callable(fn):
    print(json.dumps({{"ok": False, "error": "missing_function"}}, ensure_ascii=False))
    sys.exit(0)

cases = {json.dumps(cases_for_runner, ensure_ascii=False)}
outputs = []
for case in cases:
    result = fn(case["text"])
    outputs.append({{"id": case["id"], "result": result}})

print(json.dumps({{"ok": True, "outputs": outputs}}, ensure_ascii=False))
""".strip() + "\n"


def deterministic_verify(solution_path, iteration_dir):
    safe = {
        "solution_exists": solution_path.exists(),
        "forbidden_hidden_text_found": False,
        "runner_returncode": None,
        "runner_timeout": False,
        "structure_ok": False,
        "errors": [],
    }
    hidden_outputs = []

    if not solution_path.exists():
        safe["errors"].append("solution.py 파일이 없습니다.")
        return safe, hidden_outputs

    code_text = solution_path.read_text(encoding="utf-8")
    for forbidden in forbidden_hidden_strings():
        if forbidden and forbidden in code_text:
            safe["forbidden_hidden_text_found"] = True
            safe["errors"].append("solution.py 안에 hidden 입력 또는 기준 요약 문구가 그대로 들어 있습니다.")
            break

    runner_path = iteration_dir / "workspace" / "_runner_private.py"
    write_text(runner_path, build_runner_code(solution_path))

    try:
        proc = subprocess.run(
            [sys.executable, str(runner_path)],
            cwd=str(iteration_dir / "workspace"),
            text=True,
            capture_output=True,
            timeout=RUN_TIMEOUT_SECONDS,
            env=os.environ.copy(),
        )
        safe["runner_returncode"] = proc.returncode
    except subprocess.TimeoutExpired:
        safe["runner_timeout"] = True
        safe["errors"].append(f"solution.py 실행이 {RUN_TIMEOUT_SECONDS}초 안에 끝나지 않았습니다.")
        return safe, hidden_outputs

    if proc.returncode != 0:
        safe["errors"].append("runner subprocess가 0이 아닌 종료코드로 끝났습니다.")
        safe["stderr_tail"] = proc.stderr[-500:]
        return safe, hidden_outputs

    try:
        payload = json.loads(proc.stdout.strip())
    except json.JSONDecodeError:
        safe["errors"].append("runner 출력이 JSON이 아닙니다.")
        safe["stdout_tail"] = proc.stdout[-500:]
        return safe, hidden_outputs

    if not payload.get("ok"):
        safe["errors"].append(payload.get("error", "runner가 실패했습니다."))
        return safe, hidden_outputs

    structure_errors = []
    for item in payload.get("outputs", []):
        case_id = item.get("id")
        result = item.get("result")
        if not isinstance(result, dict):
            structure_errors.append(f"{case_id}: 반환값이 dict가 아닙니다.")
            continue
        keywords = result.get("keywords")
        summary = result.get("summary")
        if not isinstance(keywords, list) or not (2 <= len(keywords) <= 4):
            structure_errors.append(f"{case_id}: keywords는 문자열 2~4개 list여야 합니다.")
        elif not all(isinstance(value, str) and value.strip() for value in keywords):
            structure_errors.append(f"{case_id}: keywords에 빈 문자열 또는 문자열이 아닌 값이 있습니다.")
        if not isinstance(summary, str) or not summary.strip():
            structure_errors.append(f"{case_id}: summary는 비어 있지 않은 문자열이어야 합니다.")
        hidden_outputs.append({"id": case_id, "result": result})

    if len(hidden_outputs) != len(HIDDEN_CASES):
        structure_errors.append("hidden case 출력 개수가 맞지 않습니다.")

    safe["structure_ok"] = not structure_errors
    safe["hidden_case_count"] = len(HIDDEN_CASES)
    safe["output_count"] = len(hidden_outputs)
    safe["errors"].extend(structure_errors)
    safe["passed"] = (
        safe["solution_exists"]
        and not safe["forbidden_hidden_text_found"]
        and safe["runner_returncode"] == 0
        and not safe["runner_timeout"]
        and safe["structure_ok"]
        and not safe["errors"]
    )
    return safe, hidden_outputs


def build_grading_payload(hidden_outputs):
    output_by_id = {item["id"]: item["result"] for item in hidden_outputs}
    cases = []
    for case in HIDDEN_CASES:
        cases.append({
            "id": case["id"],
            "input_text": case["text"],
            "expected_keywords": case["expected_keywords"],
            "expected_summary": case["expected_summary"],
            "actual_output": output_by_id.get(case["id"]),
        })
    return {"cases": cases}


def normalize_grade(parsed_grade):
    cases = []
    for item in parsed_grade.get("cases", []):
        keyword_score = float(item.get("keyword_score", 0.0))
        summary_score = float(item.get("summary_score", 0.0))
        keyword_score = max(0.0, min(5.0, keyword_score))
        summary_score = max(0.0, min(5.0, summary_score))
        case_score = min(keyword_score, summary_score)
        cases.append({
            "id": item.get("id"),
            "keyword_score": keyword_score,
            "summary_score": summary_score,
            "case_score": case_score,
            "reason": item.get("reason", ""),
        })
    average_score = statistics.fmean([case["case_score"] for case in cases]) if cases else 0.0
    return {"cases": cases, "average_score": average_score}


def decide_pass(iteration, deterministic_safe, grade):
    thresholds = case_thresholds(iteration)
    case_scores = [case["case_score"] for case in grade.get("cases", [])]
    min_case_score = min(case_scores) if case_scores else 0.0
    average_score = float(grade.get("average_score", 0.0))
    deterministic_passed = bool(deterministic_safe.get("passed"))
    score_passed = bool(case_scores) and min_case_score >= thresholds["case_min"] and average_score >= thresholds["average_min"]
    forced_demo_failure = False
    passed = deterministic_passed and score_passed

    if iteration == 1 and DEMO_FORCE_FIRST_FAILURE and passed:
        forced_demo_failure = True
        passed = False

    return {
        "iteration": iteration,
        "threshold_mode": thresholds["mode"],
        "case_min_required": thresholds["case_min"],
        "average_min_required": thresholds["average_min"],
        "deterministic_passed": deterministic_passed,
        "min_case_score": min_case_score,
        "average_score": average_score,
        "score_passed": score_passed,
        "forced_demo_failure": forced_demo_failure,
        "passed": passed,
    }


def safe_decision_for_feedback(decision):
    return {
        "iteration": decision["iteration"],
        "threshold_mode": decision["threshold_mode"],
        "deterministic_passed": decision["deterministic_passed"],
        "min_case_score": decision["min_case_score"],
        "average_score": decision["average_score"],
        "score_passed": decision["score_passed"],
        "forced_demo_failure": decision["forced_demo_failure"],
        "passed": decision["passed"],
    }


## 7. 루프 정의

이 함수가 실제 흐름을 담당합니다. 각 단계 시작 때 지금 무엇을 하는지 평범한 말로 출력합니다.


In [ ]:
def run_demo_loop():
    run_dir = create_run_dir()
    readme_rows = []
    previous_feedback = ""
    previous_solution = None
    final_status = "failed"

    update_readme(run_dir, readme_rows, final_status="running")
    write_run_dashboard(run_dir)
    print(f"이번 실행 폴더: {run_dir}")

    for iteration in range(1, MAX_ITER + 1):
        iter_name = f"iteration-{iteration:02d}"
        iter_dir = run_dir / iter_name
        workspace_dir = iter_dir / "workspace"
        workspace_dir.mkdir(parents=True, exist_ok=True)
        solution_path = workspace_dir / SOLUTION_FILENAME
        copy_previous_solution(previous_solution, solution_path)

        display_iteration_banner(iteration)
        print_stage(f"지금 계획 중입니다. 반복 {iteration}/{MAX_ITER}")
        if iteration == 1:
            plan_result = run_agent_command(CLAUDE_CMD, build_initial_plan_prompt(), iter_dir / "01_plan.md")
        else:
            plan_result = run_agent_command(CLAUDE_CMD, build_replan_prompt(previous_feedback), iter_dir / "01_plan.md")
        plan_text = plan_result["output"] or "계획 생성 실패: 기본 규칙 기반 구현을 시도합니다."

        print_stage(f"지금 구현 중입니다. Codex가 {SOLUTION_FILENAME}만 수정합니다.")
        implementation_prompt = build_implementation_prompt(plan_text, previous_feedback)
        implementation_cmd = CODEX_IMPLEMENT_CMD_BASE + ["--cd", str(workspace_dir)]
        run_agent_command(implementation_cmd, implementation_prompt, iter_dir / "02_implementation.log", cwd=workspace_dir)

        print_stage("지금 확정 검증 중입니다. hidden 예시에서도 비지 않은 결과가 나오는지 확인합니다.")
        deterministic_safe, hidden_outputs = deterministic_verify(solution_path, iter_dir)
        write_json(iter_dir / "03_deterministic_verify.json", deterministic_safe)

        print_stage("지금 가벼운 AI로 의미 채점 중입니다. 점수는 0~5점이고 case_score = min(keyword, summary)입니다.")
        if deterministic_safe.get("passed"):
            grading_payload = build_grading_payload(hidden_outputs)
            grading_prompt = build_grading_prompt(grading_payload)
            grading_result = run_agent_command(CODEX_GRADER_CMD, grading_prompt, iter_dir / "04_ai_grade.raw.private.log")
            try:
                parsed_grade = extract_json_object(grading_result["output"])
                grade = normalize_grade(parsed_grade)
            except Exception as error:
                grade = {
                    "cases": [],
                    "average_score": 0.0,
                    "parse_error": str(error),
                    "raw_tail": grading_result["output"][-1000:],
                }
        else:
            grade = {
                "cases": [],
                "average_score": 0.0,
                "skipped": "확정 검증이 실패해서 AI 의미 채점을 건너뜁니다.",
            }
        write_json(iter_dir / "04_ai_grade.private.json", grade)

        print_stage("지금 테스트 중입니다. 노트북 Python이 점수 기준으로 합격/불합격을 결정합니다.")
        decision = decide_pass(iteration, deterministic_safe, grade)
        write_json(iter_dir / "05_decision.json", decision)
        display_iteration_result(iter_dir)

        row = {
            "iteration": iteration,
            "dir": iter_name,
            "result": "PASS" if decision["passed"] else "FAIL",
            "threshold_mode": decision["threshold_mode"],
            "diagnosis": False,
        }

        if decision["passed"]:
            final_status = "passed"
            readme_rows.append(row)
            update_readme(run_dir, readme_rows, final_status=final_status)
            write_run_dashboard(run_dir)
            display_run_dashboard(run_dir)
            print_stage("통과했습니다. 루프를 종료합니다.")
            print(f"목차: {run_dir / 'README.md'}")
            return {"run_dir": str(run_dir), "status": final_status, "iterations": iteration}

        print_stage("실패 원인을 분석 중입니다. hidden 정답은 Claude에게 주지 않습니다.")
        solution_code = read_text(solution_path)
        decision_safe = safe_decision_for_feedback(decision)
        diagnosis_prompt = build_diagnosis_prompt(solution_code, deterministic_safe, decision_safe)
        diagnosis_result = run_agent_command(CLAUDE_CMD, diagnosis_prompt, iter_dir / "06_failure_diagnosis.md")
        diagnosis_text = diagnosis_result["output"] or "진단 생성 실패. 고정 힌트를 바탕으로 다음 반복을 진행합니다."

        previous_feedback = (
            "# 다음 반복용 안전 피드백\n\n"
            "이 파일에는 hidden 입력, hidden 정답, 실제 hidden 출력, 케이스별 누락 항목을 넣지 않습니다.\n\n"
            "## 집계 판정\n"
            f"```json\n{json.dumps(decision_safe, ensure_ascii=False, indent=2)}\n```\n\n"
            "## 고정 힌트\n"
            + "\n".join(f"- {hint}" for hint in FIXED_SAFE_HINTS)
            + "\n\n## Claude 진단\n"
            + diagnosis_text
        )
        write_text(iter_dir / "feedback_for_next_iter.md", previous_feedback)

        previous_solution = solution_path
        row["diagnosis"] = True
        readme_rows.append(row)
        update_readme(run_dir, readme_rows, final_status="running")
        write_run_dashboard(run_dir)

    final_status = "max-iter-reached"
    update_readme(run_dir, readme_rows, final_status=final_status)
    write_run_dashboard(run_dir)
    display_run_dashboard(run_dir)
    print_stage(f"{MAX_ITER}회 안에 통과하지 못했습니다. 자동 루프를 멈춥니다.")
    print(f"목차: {run_dir / 'README.md'}")
    return {"run_dir": str(run_dir), "status": final_status, "iterations": MAX_ITER}


print("run_demo_loop() 준비 완료")


In [ ]:
for case in HIDDEN_CASES:
    print("=" * 60)
    print(case["id"])
    print("입력:", case["text"])
    print("기준 키워드:", ", ".join(case["expected_keywords"]))
    print("기준 요약:", case["expected_summary"])

## 8. 실행

기본값은 실행하지 않음입니다. 실제로 흐름을 보고 싶을 때만 `RUN_LOOP = False`로 바꾸고 이 셀을 실행하세요.


In [ ]:
RUN_LOOP = False

if RUN_LOOP:
    result = run_demo_loop()
    print(json.dumps(result, ensure_ascii=False, indent=2))
else:
    print("RUN_LOOP가 False라서 실행하지 않았습니다. 실행하려면 True로 바꾸세요.")


## 9. 실행 결과 HTML 대시보드

이미 실행한 최신 결과를 다시 보고 싶으면 아래 셀만 실행하세요. `loop_demo_runs`의 가장 최신 실행 폴더를 읽어서 HTML 카드로 보여줍니다.


In [ ]:
display_run_dashboard()